# 03 - Casson--Gordon invariants and genus obstructions

This notebook follows the implemented double-cover formulas for supported `(2,q)`-cable sums. It first separates the exact contributions to a Casson--Gordon invariant and then shows how those values enter a finite, auditable Gilmer obstruction search.

## Learning objectives

- compute Casson--Gordon signature and nullity for a supported cable;
- inspect pattern, companion, and connected-sum contributions;
- work with the distinguished diagonal linking form;
- recognize isotropic lines and metabolizers; and
- interpret a successful or inconclusive four-genus obstruction correctly.

The character parameters in this API are integers in a geometrically distinguished lens-space basis. They are deliberately **not** `Character` objects, whose coordinates use a computed Smith basis.

## 1. Setup

In [ ]:
from pathlib import Path
from dataclasses import asdict
import sys

repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
source_directory = repository_root / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from gaknot import (
    GeneralizedAlgebraicKnot,
    PrimeDiagonalLinkingForm,
)

trefoil = GeneralizedAlgebraicKnot.torus_knot(2, 3)
cable = trefoil.cable(2, 5)
print(cable)

## 2. The `(2,q)` cabling formula

For nonzero `a mod q`, the supported formula is

`sigma(K(2,q), chi_a) = -q + 2*a*(q-a)/q + 2*sigma_K(exp(2*pi*i*a/q))`.

The first two terms form the torus-pattern contribution. The final term is twice the ordinary Levine--Tristram signature of the companion at the selected root of unity.

In [ ]:
cg = cable.casson_gordon(1)

print("normalized parameters:", cg.character_parameters)
print("pattern contribution:", cg.pattern_signature)
print("twice the companion contribution:", cg.satellite_signature)
print("total sigma:", cg.sigma)
print("total eta:", cg.eta)

The immutable result contains one `CassonGordonSummand` for every visible connected-sum component. Inspecting the record makes the formula auditable instead of exposing only its final sum.

In [ ]:
asdict(cg.summands[0])

The dataclass dictionary stores the raw signed companion signature; the derived property `satellite_signature` multiplies it by two. A parameter is always reduced modulo `q`.

In [ ]:
same_character = cable.casson_gordon(6)
trivial_character = cable.casson_gordon(0)

print("6 and 1 define the same parameter modulo 5:", same_character == cg)
print("trivial-character sigma:", trivial_character.sigma)
print("trivial-character eta:", trivial_character.eta)

The trivial character is a separate case. Substituting `a=0` into the displayed nontrivial formula would incorrectly leave the `-q` term, so the implementation handles it explicitly.

## 3. Connected sums and nullity

Signature is additive. If `r` component restrictions are nontrivial, connected sum contributes an additional `r-1` to nullity, beyond the component nullities.

In [ ]:
formal_slice_pair = cable + (-cable)
pair_invariant = formal_slice_pair.casson_gordon([1, 1])

print("summand signatures:", tuple(s.sigma for s in pair_invariant.summands))
print("total signature:", pair_invariant.sigma)
print("total nullity:", pair_invariant.eta)

The opposite component signs cancel the signatures, while the two nontrivial restrictions leave the connected-sum nullity correction equal to one. This illustrates why preserving both structural summands of `K + (-K)` is useful.

## 4. The distinguished linking form

For the supported family, each component contributes a prime-order cyclic group in a distinguished geometric basis. `PrimeDiagonalLinkingForm` stores the self-pairing numerators and prime orders exactly.

In [ ]:
linking_form = PrimeDiagonalLinkingForm.from_knot(formal_slice_pair)

print(linking_form)
print("orders:", linking_form.orders)
print("diagonal numerators:", linking_form.coefficients)
print("primary primes:", linking_form.primary_primes)

The opposite knot signs produce opposite diagonal coefficients. Hence the diagonal vector `(1,1)` is self-annihilating.

In [ ]:
vector = (1, 1)

print("self-pairing:", linking_form.pairing(vector, vector))
print("isotropic:", linking_form.is_isotropic(vector))
print("generates a metabolizer:", linking_form.is_metabolizer([vector]))
print("projective isotropic representatives:",
      list(linking_form.projective_isotropic_elements(5)))

A metabolizer must be both totally isotropic and half-dimensional in every primary part. Enumerating only projective representatives avoids checking all `p-1` nonzero scalar multiples of the same line.

## 5. Gilmer's inequality as a sufficient obstruction

The implemented search uses

`|sigma(K, chi_x) + sigma_K| <= eta(K, chi_x) + 4*g + 1`.

If the inequality is forced to fail on every relevant isotropic line, the knot cannot bound a locally flat genus-`g` surface in the four-ball.

In [ ]:
obstruction = cable.gilmer_genus_obstruction(0)

print("tested genus:", obstruction.tested_genus)
print("classical signature:", obstruction.classical_signature)
print("certified:", obstruction.certified)
print("certified lower bound:", obstruction.lower_bound)
print("successful primary orders:", obstruction.successful_primes)

Each primary check retains the size of the search and, when successful, a sample violating character. These diagnostics are useful when reproducing a proof or investigating why a proposed obstruction is inconclusive.

In [ ]:
for check in obstruction.primary_checks:
    print("primary check:")
    print(asdict(check))

Interpretation is one-sided:

- `certified=True` proves `g_4^top(K) > g`, so `lower_bound` is `g+1`.
- `certified=False` means only that this particular sufficient search did not certify the bound.

An inconclusive result does **not** construct a surface and does not prove `g_4^top(K) <= g`.

## 6. Support boundary

The current Casson--Gordon implementation accepts signed sums whose outermost pattern has winding number two and prime order `q`. A general cable, composite `q`, or a coordinate supplied in the public Smith basis needs additional mathematical input. Unsupported cases raise an exception rather than silently identifying incompatible bases.

## Exercises

1. Compute the invariant for parameters `a=1,2,3,4` and look for the expected symmetry.
2. Compare `cable.casson_gordon(1)` with `(-cable).casson_gordon(1)` component by component.
3. Build `PrimeDiagonalLinkingForm([5,5], [1,-1])` directly and verify the metabolizer calculation.
4. Run the genus obstruction at `g=1`. If it is inconclusive, explain precisely what can and cannot be concluded.